[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/07.%20Clase%207/07Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F07.%20Clase%207%2F07Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 7: Comparación de estimadores y análisis de riesgo

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Simular trayectorias de precios con tres modelos: **normal**, **histograma**, **KDE**.
- Calcular **VaR** y **CVaR** (Expected Shortfall) por simulación.
- Comparar estimadores **clásicos vs. robustos** en la simulación.
- Visualizar distribuciones de precios finales y regiones de pérdida.
- Entender por qué el CVaR es una medida **coherente** de riesgo.

---

## Introducción teórica

En las Clases 5 y 6 introdujimos estimadores robustos para la covarianza (Shrunk/Ledoit-Wolf) y la media (Huber). En esta clase:

1. **Simulamos trayectorias de precios** usando tres modelos: normal, histograma empírico y estimación de densidad por kernel (KDE).
2. **Comparamos los estimadores** clásicos vs. robustos en la simulación.
3. **Medimos el riesgo** mediante VaR y CVaR (Expected Shortfall).

### Simulación Monte Carlo de precios

Bajo el modelo GBM, el precio futuro a $T$ días es:

$$
S_T = S_0 \exp\!\left(\sum_{t=1}^{T} r_t\right)
$$

donde los rendimientos $r_t$ se generan según distintos modelos:

| Modelo | Descripción | Ventaja | Limitación |
|--------|-------------|---------|------------|
| **Normal** | $r_t \sim \mathcal{N}(\mu, \sigma^2)$ | Simple, analítico | Ignora colas pesadas |
| **Histograma** | Muestreo de la distribución empírica | Captura forma real | Discreto, ruidoso |
| **KDE** | Suavizado de la distribución empírica | Continuo, flexible | Sensible al bandwidth |

### Medidas de riesgo

- **VaR** (Value at Risk) al nivel $\alpha$: pérdida máxima con probabilidad $1 - \alpha$.
- **CVaR** (Conditional VaR / Expected Shortfall): pérdida esperada *dado que* se excede el VaR. Es una medida **coherente** de riesgo (Artzner et al., 1999).

## 1. Datos históricos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.neighbors import KernelDensity
import statsmodels.api as sm

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
ticker = "AAPL"
data = yf.download(ticker, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
closes.plot(ax=axes[0], title=f"{ticker}: Precio de cierre")
axes[0].set_ylabel("USD")
daily_returns.plot(ax=axes[1], title=f"{ticker}: Rendimientos logarítmicos")
axes[1].set_ylabel("Rendimiento")
plt.tight_layout()

---
## 2. Modelo 1: Simulación con distribución normal

Bajo el supuesto de log-normalidad (GBM), los rendimientos diarios se distribuyen:

$$
r_t \sim \mathcal{N}(\hat{\mu}, \hat{\sigma}^2)
$$

Generamos $N$ trayectorias de $T$ días hacia el futuro.

In [ ]:
np.random.seed(42)
S0 = closes.iloc[-1]
mu = daily_returns.mean()
sigma = daily_returns.std()
T = 180       # Días a simular
N_traj = 500  # Número de trayectorias

print(f"Precio inicial: ${S0:.2f}")
print(f"μ diario: {mu:.6f}")
print(f"σ diario: {sigma:.6f}")

# Simular rendimientos normales
sim_returns_normal = np.random.normal(mu, sigma, size=(T, N_traj))
sim_prices_normal = S0 * np.exp(np.cumsum(sim_returns_normal, axis=0))

# Gráfico
dates_future = pd.date_range(closes.index[-1] + pd.Timedelta(days=1), periods=T, freq="B")

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(closes.index, closes.values, "k-", linewidth=2, label="Histórico")
ax.plot(dates_future, sim_prices_normal, alpha=0.05, color="steelblue")
ax.plot(dates_future, np.median(sim_prices_normal, axis=1), "r--", linewidth=2, label="Mediana simulada")
ax.set_title(f"Monte Carlo — Modelo Normal ({N_traj} trayectorias, {T} días)")
ax.set_ylabel("Precio (USD)")
ax.legend()
plt.tight_layout()

---
## 3. Modelo 2: Simulación con histograma empírico

En lugar de asumir normalidad, muestreamos directamente de la **distribución empírica** de los rendimientos históricos. Esto captura la forma real de la distribución (colas pesadas, asimetría) sin imponer un modelo paramétrico.

In [ ]:
# Histograma empírico como distribución discreta
values, bin_edges = np.histogram(daily_returns, bins=200)
weights = values / values.sum()
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Muestrear del histograma
sim_returns_hist = np.random.choice(bin_centers, size=(T, N_traj), p=weights)
sim_prices_hist = S0 * np.exp(np.cumsum(sim_returns_hist, axis=0))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(closes.index, closes.values, "k-", linewidth=2, label="Histórico")
ax.plot(dates_future, sim_prices_hist, alpha=0.05, color="tab:orange")
ax.plot(dates_future, np.median(sim_prices_hist, axis=1), "r--", linewidth=2, label="Mediana simulada")
ax.set_title(f"Monte Carlo — Histograma empírico ({N_traj} trayectorias)")
ax.set_ylabel("Precio (USD)")
ax.legend()
plt.tight_layout()

---
## 4. Modelo 3: Estimación de densidad por kernel (KDE)

La **KDE** suaviza la distribución empírica mediante una suma de kernels (típicamente gaussianos):

$$
\hat{f}(r) = \frac{1}{Th}\sum_{t=1}^{T} K\!\left(\frac{r - r_t}{h}\right)
$$

donde $h$ es el **bandwidth** (ancho de banda) que controla el suavizado. A diferencia del histograma, la KDE produce una distribución **continua**.

In [ ]:
# Ajustar KDE
kde = KernelDensity(kernel="gaussian", bandwidth=0.002).fit(daily_returns.values.reshape(-1, 1))

# Muestrear de la KDE
sim_returns_kde = kde.sample(n_samples=T * N_traj, random_state=42).reshape(T, N_traj)
sim_prices_kde = S0 * np.exp(np.cumsum(sim_returns_kde, axis=0))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(closes.index, closes.values, "k-", linewidth=2, label="Histórico")
ax.plot(dates_future, sim_prices_kde, alpha=0.05, color="tab:green")
ax.plot(dates_future, np.median(sim_prices_kde, axis=1), "r--", linewidth=2, label="Mediana simulada")
ax.set_title(f"Monte Carlo — KDE ({N_traj} trayectorias)")
ax.set_ylabel("Precio (USD)")
ax.legend()
plt.tight_layout()

---
## 5. Comparación de los tres modelos

### Distribución de precios finales

Comparamos la distribución del precio simulado a $T$ días para cada modelo:

In [ ]:
final_prices = pd.DataFrame({
    "Normal": sim_prices_normal[-1],
    "Histograma": sim_prices_hist[-1],
    "KDE": sim_prices_kde[-1]
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, color in zip(axes, final_prices.columns, ["steelblue", "tab:orange", "tab:green"]):
    sns.histplot(final_prices[col], bins=50, stat="density", kde=True, ax=ax, color=color, alpha=0.6)
    ax.axvline(S0, color="red", linestyle="--", label=f"Precio actual: ${S0:.0f}")
    ax.set_title(f"Modelo {col}")
    ax.set_xlabel("Precio final (USD)")
    ax.legend(fontsize=9)
plt.suptitle(f"Distribución de precios a {T} días ({N_traj} simulaciones)", fontsize=14)
plt.tight_layout()

In [ ]:
# Estadísticas comparativas
stats_df = final_prices.describe().T
stats_df["Skew"] = final_prices.skew()
stats_df["Kurtosis"] = final_prices.kurtosis()
stats_df[["mean", "std", "min", "25%", "50%", "75%", "max", "Skew", "Kurtosis"]]

---
## 6. Medidas de riesgo: VaR y CVaR

### Value at Risk (VaR)

El **VaR** al nivel de confianza $\alpha$ (e.g., 95%) es la pérdida máxima que no se excede con probabilidad $\alpha$:

$$
\text{VaR}_\alpha = -\text{Percentil}_{1-\alpha}(\text{P\&L})
$$

### Conditional VaR (CVaR / Expected Shortfall)

El **CVaR** es la pérdida esperada condicional a que se exceda el VaR:

$$
\text{CVaR}_\alpha = -\mathbb{E}[\text{P\&L} \mid \text{P\&L} \leq -\text{VaR}_\alpha]
$$

El CVaR es siempre mayor o igual que el VaR y es una medida **coherente** de riesgo (Artzner et al., 1999), lo que significa que satisface subaditividad: el riesgo de un portafolio es menor o igual a la suma de los riesgos individuales.

In [ ]:
alpha = 0.95  # Nivel de confianza

# P&L como retorno porcentual
pnl = {name: (prices - S0) / S0 for name, prices in
       zip(["Normal", "Histograma", "KDE"],
           [sim_prices_normal[-1], sim_prices_hist[-1], sim_prices_kde[-1]])}

risk_metrics = []
for model, losses in pnl.items():
    var = -np.percentile(losses, (1 - alpha) * 100)
    cvar = -losses[losses <= -var].mean() if (losses <= -var).any() else var
    risk_metrics.append({
        "Modelo": model,
        f"VaR ({alpha:.0%})": f"{var:.2%}",
        f"CVaR ({alpha:.0%})": f"{cvar:.2%}",
        "Media P&L": f"{losses.mean():.2%}",
    })

risk_df = pd.DataFrame(risk_metrics).set_index("Modelo")
risk_df

In [ ]:
# Visualización del VaR y CVaR
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (model, losses), color in zip(axes, pnl.items(), ["steelblue", "tab:orange", "tab:green"]):
    var_val = -np.percentile(losses, (1 - alpha) * 100)
    
    sns.histplot(losses, bins=50, stat="density", ax=ax, color=color, alpha=0.5)
    ax.axvline(-var_val, color="red", linewidth=2, linestyle="--", label=f"VaR {alpha:.0%} = {var_val:.1%}")
    ax.fill_betweenx([0, ax.get_ylim()[1]], ax.get_xlim()[0], -var_val,
                     alpha=0.15, color="red", label=f"CVaR region")
    ax.set_title(f"{model}")
    ax.set_xlabel("P&L (%)")
    ax.legend(fontsize=9)
plt.suptitle(f"VaR y CVaR al {alpha:.0%} — {T} días, {N_traj} simulaciones", fontsize=14)
plt.tight_layout()

---
## 7. Estimadores clásicos vs. robustos

Comparamos la simulación usando:
- **Clásico**: media muestral + desviación estándar muestral
- **Robusto**: media de Huber + escala de Huber (Clase 6)

In [ ]:
huber = sm.robust.scale.Huber()
mu_huber, sigma_huber = huber(daily_returns.values.flatten())

print(f"Estimadores clásicos:  μ = {mu:.6f},  σ = {sigma:.6f}")
print(f"Estimadores Huber:     μ = {mu_huber:.6f},  σ = {sigma_huber:.6f}")

# Simular con estimadores robustos
sim_returns_robust = np.random.normal(mu_huber, sigma_huber, size=(T, N_traj))
sim_prices_robust = S0 * np.exp(np.cumsum(sim_returns_robust, axis=0))

# Comparar distribuciones finales
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(sim_prices_normal[-1], bins=50, stat="density", label="Normal (clásico)",
             color="steelblue", alpha=0.5, ax=ax)
sns.histplot(sim_prices_robust[-1], bins=50, stat="density", label="Normal (Huber)",
             color="tab:red", alpha=0.5, ax=ax)
ax.axvline(S0, color="black", linestyle="--", label=f"Precio actual: ${S0:.0f}")
ax.set_title("Distribución de precio final: estimadores clásicos vs. Huber")
ax.set_xlabel("Precio final (USD)")
ax.legend()
plt.tight_layout()

In [ ]:
# VaR comparativo
var_classic = -np.percentile((sim_prices_normal[-1] - S0) / S0, (1-alpha)*100)
var_robust  = -np.percentile((sim_prices_robust[-1] - S0) / S0, (1-alpha)*100)
print(f"VaR {alpha:.0%} — Clásico: {var_classic:.2%}")
print(f"VaR {alpha:.0%} — Huber:   {var_robust:.2%}")

---

## Navegación del curso

← **Anterior**: Clase 6: Media robusta y RNG  
→ **Siguiente**: Clase 8: Resumen Parte 1

---

## 8. Referencias bibliográficas

- **Artzner, P., Delbaen, F., Eber, J.-M. & Heath, D.** (1999). Coherent Measures of Risk. *Mathematical Finance*, 9(3), 203–228.
- **Cont, R.** (2001). Empirical Properties of Asset Returns. *Quantitative Finance*, 1(2), 223–236.
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer. — Cap. 1–3.
- **Huber, P. J.** (1964). Robust Estimation of a Location Parameter. *The Annals of Mathematical Statistics*, 35(1), 73–101.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 22: Value at Risk.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press.
- **McNeil, A. J., Frey, R. & Embrechts, P.** (2015). *Quantitative Risk Management* (2nd ed.). Princeton University Press. — Cap. 2: Basic Concepts in Risk Management.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. — Cap. 7: Value at Risk.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.